In [ ]:
#Import required libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import glob
import json

print("Libraries imported successfully")


In [ ]:
#Define the same model architecture as train
class SimpsonsClassifier(nn.Module):
    def __init__(self, num_classes):
        super(SimpsonsClassifier, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))  # matches train.ipynb
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512 * 4 * 4, 1024),  # matches train.ipynb
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

print('Model architecture defined')


In [ ]:
def infer(data_dir, model_path):
    """
    Load model and generate predictions on images in data_dir.
    Saves results to results.json.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')


    print('Loading model...')
    checkpoint = torch.load(model_path, map_location=device)
    num_classes = checkpoint['num_classes']
    idx_to_class = checkpoint['idx_to_class']


    model = SimpsonsClassifier(num_classes)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    print(f'Model loaded — {num_classes} classes')

    #Transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    #Collect images, use os.walk only (no duplicate from top-level glob)
    print('Collecting images...')
    image_files = []
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_files.append(os.path.join(root, file))
    print(f'Found {len(image_files)} images')

    #Prediction step
    print('Generating predictions...')
    results = {}
    with torch.no_grad():
        for image_path in image_files:
            try:
                image = Image.open(image_path).convert('RGB')
                image_tensor = transform(image).unsqueeze(0).to(device)
                output = model(image_tensor)
                predicted_idx = output.argmax(dim=1).item()
                predicted_class = idx_to_class[predicted_idx]
                results[os.path.basename(image_path)] = predicted_class
            except Exception as e:
                print(f'Error processing {image_path}: {e}')
                results[os.path.basename(image_path)] = 'unknown'

    #Save results
    with open('results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Saved results.json — {len(results)} predictions')
    return results

print('Inference function defined')


In [ ]:
#Update these paths before running
data_dir = '/content/archive/characters_train/test_data'  # directory with test images
model_path = '/content/model.pth'

results = infer(data_dir, model_path)
